# Cleaning ACS Data

**Goal:**  
Prepare the ACS data so it can be used later in the housing affordability analysis.

**Plan:**
1. Load the raw ACS files
2. Combine the yearly files into one dataset
3. Check the combined data
4. Rename and organize the columns
5. Fix data types
6. Clean county and FIPS columns
7. Create useful housing variables
8. Create ACS age group data
9. Finalize ACS age group data using CBSA codes
10. Finalize ACS county-year data
11. Save the cleaned ACS datasets


In [33]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

### Step 1: Loading Raw ACS Files

In [34]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

acs_path = project_root / "data" / "ACS"
clean_path = project_root / "clean_data"

clean_path.mkdir(exist_ok=True)

acs_files = sorted([
    file for file in acs_path.glob("acs*.csv")
    if file.name[3:7].isdigit() and "-" not in file.stem
])

if len(acs_files) == 0:
    raise FileNotFoundError("No ACS files found. Make sure files like acs2005.csv are in data/ACS.")

print("CSV files found:")
for file in acs_files:
    print(file.name)


CSV files found:
acs2005.csv
acs2006.csv
acs2007.csv
acs2008.csv
acs2009.csv
acs2010.csv
acs2011.csv
acs2012.csv
acs2013.csv
acs2018.csv
acs2019.csv
acs2021.csv
acs2022.csv
acs2023.csv


### Step 2: Combining the Files

In [35]:
dfs = []

for file in acs_files:
    df = pd.read_csv(file, dtype=str)
    year = int(file.name[3:7])
    df["year"] = year
    dfs.append(df)

acs = pd.concat(dfs, ignore_index=True)

acs.head()


,NAME,B25007_001E,B25007_002E,B25007_003E,B25007_004E,B25007_005E,B25007_006E,B25007_007E,B25007_008E,B25007_009E,...,B25007_015E,B25007_016E,B25007_017E,state,county,year,B25007_018E,B25007_019E,B25007_020E,B25007_021E
0,"Atlantic County, New Jersey",101584,68970,980,7452,14735,16400,6754,5659,7839,...,7108,6368,1624,34,001,2005,NaN,NaN,NaN,NaN
1,"Bergen County, New Jersey",332223,225175,763,13773,50079,54557,24781,20251,28909,...,25357,20069,8337,34,003,2005,NaN,NaN,NaN,NaN
2,"Burlington County, New Jersey",164046,129201,1360,14486,28484,34191,11863,10931,14120,...,6761,5952,2318,34,005,2005,NaN,NaN,NaN,NaN
3,"Camden County, New Jersey",190659,130576,1141,14690,26698,33575,14054,10884,14414,...,13478,10447,3196,34,007,2005,NaN,NaN,NaN,NaN
4,"Cape May County, New Jersey",43616,34101,257,2457,5958,7586,3447,3174,5332,...,1614,1190,680,34,009,2005,NaN,NaN,NaN,NaN


In [36]:
acs.shape

(294, 25)

This shows that the ACS files were loaded correctly. Each row is one county in one year.


### Step 3: Checking the Combined Data

In [37]:
# Check data types and non-null counts
acs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294 entries, 0 to 293
Data columns (total 25 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   NAME         294 non-null    object
 1   B25007_001E  294 non-null    object
 2   B25007_002E  294 non-null    object
 3   B25007_003E  294 non-null    object
 4   B25007_004E  294 non-null    object
 5   B25007_005E  294 non-null    object
 6   B25007_006E  294 non-null    object
 7   B25007_007E  294 non-null    object
 8   B25007_008E  294 non-null    object
 9   B25007_009E  294 non-null    object
 10  B25007_010E  294 non-null    object
 11  B25007_011E  294 non-null    object
 12  B25007_012E  294 non-null    object
 13  B25007_013E  294 non-null    object
 14  B25007_014E  294 non-null    object
 15  B25007_015E  294 non-null    object
 16  B25007_016E  294 non-null    object
 17  B25007_017E  294 non-null    object
 18  state        294 non-null    object
 19  county       294 non-null    

In [38]:
# Check column names
acs.columns

Index(['NAME', 'B25007_001E', 'B25007_002E', 'B25007_003E', 'B25007_004E',
       'B25007_005E', 'B25007_006E', 'B25007_007E', 'B25007_008E',
       'B25007_009E', 'B25007_010E', 'B25007_011E', 'B25007_012E',
       'B25007_013E', 'B25007_014E', 'B25007_015E', 'B25007_016E',
       'B25007_017E', 'state', 'county', 'year', 'B25007_018E', 'B25007_019E',
       'B25007_020E', 'B25007_021E'],
      dtype='object')

In [39]:
# Check missing values
acs.isnull().sum()

NAME             0
B25007_001E      0
B25007_002E      0
B25007_003E      0
B25007_004E      0
B25007_005E      0
B25007_006E      0
B25007_007E      0
B25007_008E      0
B25007_009E      0
B25007_010E      0
B25007_011E      0
B25007_012E      0
B25007_013E      0
B25007_014E      0
B25007_015E      0
B25007_016E      0
B25007_017E      0
state            0
county           0
year             0
B25007_018E    189
B25007_019E    189
B25007_020E    189
B25007_021E    189
dtype: int64

### Step 4: Renaming Columns

The ACS columns use Census codes, so I renamed them to make the data easier to read and use.

In [40]:
acs = acs.rename(columns={
    "NAME": "county_name",
    "B25007_001E": "total_occupied_units",
    "B25007_002E": "owner_occupied_total",
    "B25007_003E": "owner_occupied_hh_15_24",
    "B25007_004E": "owner_occupied_hh_25_34",
    "B25007_005E": "owner_occupied_hh_35_44",
    "B25007_006E": "owner_occupied_hh_45_54",
    "B25007_007E": "owner_occupied_hh_55_59",
    "B25007_008E": "owner_occupied_hh_60_64",
    "B25007_009E": "owner_occupied_hh_65_74",
    "B25007_010E": "owner_occupied_hh_75_84",
    "B25007_011E": "owner_occupied_hh_85_plus",
    "B25007_012E": "renter_occupied_total",
    "B25007_013E": "renter_occupied_hh_15_24",
    "B25007_014E": "renter_occupied_hh_25_34",
    "B25007_015E": "renter_occupied_hh_35_44",
    "B25007_016E": "renter_occupied_hh_45_54",
    "B25007_017E": "renter_occupied_hh_55_59",
    "B25007_018E": "renter_occupied_hh_60_64",
    "B25007_019E": "renter_occupied_hh_65_74",
    "B25007_020E": "renter_occupied_hh_75_84",
    "B25007_021E": "renter_occupied_hh_85_plus",
    "state": "state_fips",
    "county": "county_fips"
})

acs.head()


,county_name,total_occupied_units,owner_occupied_total,owner_occupied_hh_15_24,owner_occupied_hh_25_34,owner_occupied_hh_35_44,owner_occupied_hh_45_54,owner_occupied_hh_55_59,owner_occupied_hh_60_64,owner_occupied_hh_65_74,...,renter_occupied_hh_35_44,renter_occupied_hh_45_54,renter_occupied_hh_55_59,state_fips,county_fips,year,renter_occupied_hh_60_64,renter_occupied_hh_65_74,renter_occupied_hh_75_84,renter_occupied_hh_85_plus
0,"Atlantic County, New Jersey",101584,68970,980,7452,14735,16400,6754,5659,7839,...,7108,6368,1624,34,001,2005,NaN,NaN,NaN,NaN
1,"Bergen County, New Jersey",332223,225175,763,13773,50079,54557,24781,20251,28909,...,25357,20069,8337,34,003,2005,NaN,NaN,NaN,NaN
2,"Burlington County, New Jersey",164046,129201,1360,14486,28484,34191,11863,10931,14120,...,6761,5952,2318,34,005,2005,NaN,NaN,NaN,NaN
3,"Camden County, New Jersey",190659,130576,1141,14690,26698,33575,14054,10884,14414,...,13478,10447,3196,34,007,2005,NaN,NaN,NaN,NaN
4,"Cape May County, New Jersey",43616,34101,257,2457,5958,7586,3447,3174,5332,...,1614,1190,680,34,009,2005,NaN,NaN,NaN,NaN


Each row is one county-year. Owner and renter columns count housing units by the age of the householder. `hh` means householder.

### Step 5: Fixing Data Types

In [41]:
acs.dtypes

county_name                   object
total_occupied_units          object
owner_occupied_total          object
owner_occupied_hh_15_24       object
owner_occupied_hh_25_34       object
owner_occupied_hh_35_44       object
owner_occupied_hh_45_54       object
owner_occupied_hh_55_59       object
owner_occupied_hh_60_64       object
owner_occupied_hh_65_74       object
owner_occupied_hh_75_84       object
owner_occupied_hh_85_plus     object
renter_occupied_total         object
renter_occupied_hh_15_24      object
renter_occupied_hh_25_34      object
renter_occupied_hh_35_44      object
renter_occupied_hh_45_54      object
renter_occupied_hh_55_59      object
state_fips                    object
county_fips                   object
year                           int64
renter_occupied_hh_60_64      object
renter_occupied_hh_65_74      object
renter_occupied_hh_75_84      object
renter_occupied_hh_85_plus    object
dtype: object

The count columns need to be numeric so they can be used for calculations later.

In [42]:
possible_count_cols = [
    "total_occupied_units",
    "owner_occupied_total",
    "owner_occupied_hh_15_24",
    "owner_occupied_hh_25_34",
    "owner_occupied_hh_35_44",
    "owner_occupied_hh_45_54",
    "owner_occupied_hh_55_59",
    "owner_occupied_hh_60_64",
    "owner_occupied_hh_65_74",
    "owner_occupied_hh_75_84",
    "owner_occupied_hh_85_plus",
    "renter_occupied_total",
    "renter_occupied_hh_15_24",
    "renter_occupied_hh_25_34",
    "renter_occupied_hh_35_44",
    "renter_occupied_hh_45_54",
    "renter_occupied_hh_55_59",
    "renter_occupied_hh_60_64",
    "renter_occupied_hh_65_74",
    "renter_occupied_hh_75_84",
    "renter_occupied_hh_85_plus",
]

count_cols = [col for col in possible_count_cols if col in acs.columns]
missing_count_cols = [col for col in possible_count_cols if col not in acs.columns]

for col in count_cols:
    acs[col] = pd.to_numeric(acs[col], errors="coerce")

acs["year"] = pd.to_numeric(acs["year"], errors="coerce").astype(int)
acs["state_fips"] = acs["state_fips"].astype(str).str.zfill(2)
acs["county_fips"] = acs["county_fips"].astype(str).str.zfill(3)

print("Missing count columns:", missing_count_cols)

acs.dtypes


Missing count columns: []


county_name                    object
total_occupied_units            int64
owner_occupied_total            int64
owner_occupied_hh_15_24         int64
owner_occupied_hh_25_34         int64
owner_occupied_hh_35_44         int64
owner_occupied_hh_45_54         int64
owner_occupied_hh_55_59         int64
owner_occupied_hh_60_64         int64
owner_occupied_hh_65_74         int64
owner_occupied_hh_75_84         int64
owner_occupied_hh_85_plus       int64
renter_occupied_total           int64
renter_occupied_hh_15_24        int64
renter_occupied_hh_25_34        int64
renter_occupied_hh_35_44        int64
renter_occupied_hh_45_54        int64
renter_occupied_hh_55_59        int64
state_fips                     object
county_fips                    object
year                            int64
renter_occupied_hh_60_64      float64
renter_occupied_hh_65_74      float64
renter_occupied_hh_75_84      float64
renter_occupied_hh_85_plus    float64
dtype: object

### Step 6: Cleaning County and FIPS Columns

This step cleans the county name and creates a full county FIPS code so the ACS data can merge with the other datasets later.

In [43]:
acs["county_clean"] = (
    acs["county_name"]
    .str.replace(" County, New Jersey", "", regex=False)
    .str.strip()
)

acs["county_fips_full"] = acs["state_fips"].str.zfill(2) + acs["county_fips"].str.zfill(3)

acs[["county_name", "county_clean", "state_fips", "county_fips", "county_fips_full", "year"]].head()

,county_name,county_clean,state_fips,county_fips,county_fips_full,year
0,"Atlantic County, New Jersey",Atlantic,34,001,34001,2005
1,"Bergen County, New Jersey",Bergen,34,003,34003,2005
2,"Burlington County, New Jersey",Burlington,34,005,34005,2005
3,"Camden County, New Jersey",Camden,34,007,34007,2005
4,"Cape May County, New Jersey",Cape May,34,009,34009,2005


### Step 7: Creating Useful ACS Variables

This step creates simple rates that are easier to compare across counties and years.

In [44]:
acs["owner_rate"] = acs["owner_occupied_total"] / acs["total_occupied_units"]
acs["renter_rate"] = acs["renter_occupied_total"] / acs["total_occupied_units"]

acs[[
    "county_clean",
    "year",
    "total_occupied_units",
    "owner_occupied_total",
    "renter_occupied_total",
    "owner_rate",
    "renter_rate"
]].head()

,county_clean,year,total_occupied_units,owner_occupied_total,renter_occupied_total,owner_rate,renter_rate
0,Atlantic,2005,101584,68970,32614,0.678946,0.321054
1,Bergen,2005,332223,225175,107048,0.677783,0.322217
2,Burlington,2005,164046,129201,34845,0.787590,0.212410
3,Camden,2005,190659,130576,60083,0.684867,0.315133
4,Cape May,2005,43616,34101,9515,0.781846,0.218154


### Step 8: Creating ACS Age Group Data

This step keeps the ACS age groups instead of estimating generations. Each row becomes one county, year, and age group.


In [45]:
age_group_cols = {
    "owner_occupied_hh_15_24": ("owner", "15-24"),
    "owner_occupied_hh_25_34": ("owner", "25-34"),
    "owner_occupied_hh_35_44": ("owner", "35-44"),
    "owner_occupied_hh_45_54": ("owner", "45-54"),
    "owner_occupied_hh_55_59": ("owner", "55-59"),
    "owner_occupied_hh_60_64": ("owner", "60-64"),
    "owner_occupied_hh_65_74": ("owner", "65-74"),
    "owner_occupied_hh_75_84": ("owner", "75-84"),
    "owner_occupied_hh_85_plus": ("owner", "85+"),
    "renter_occupied_hh_15_24": ("renter", "15-24"),
    "renter_occupied_hh_25_34": ("renter", "25-34"),
    "renter_occupied_hh_35_44": ("renter", "35-44"),
    "renter_occupied_hh_45_54": ("renter", "45-54"),
    "renter_occupied_hh_55_59": ("renter", "55-59"),
    "renter_occupied_hh_60_64": ("renter", "60-64"),
    "renter_occupied_hh_65_74": ("renter", "65-74"),
    "renter_occupied_hh_75_84": ("renter", "75-84"),
    "renter_occupied_hh_85_plus": ("renter", "85+"),
}

age_group_cols = {
    col: values
    for col, values in age_group_cols.items()
    if col in acs.columns
}

age_group_rows = []

id_cols = [
    "county_clean",
    "county_fips_full",
    "year",
    "total_occupied_units",
]

for _, row in acs.iterrows():
    base_values = {col: row[col] for col in id_cols}

    for source_col, (tenure, age_group) in age_group_cols.items():
        new_row = base_values.copy()
        new_row.update({
            "age_group": age_group,
            "tenure": tenure,
            "housing_units_estimated": row[source_col],
        })
        age_group_rows.append(new_row)

acs_age_group_long = pd.DataFrame(age_group_rows)

acs_age_group_long = (
    acs_age_group_long
    .groupby(id_cols + ["age_group", "tenure"], as_index=False)["housing_units_estimated"]
    .sum()
)

acs_age_group = (
    acs_age_group_long
    .pivot_table(
        index=id_cols + ["age_group"],
        columns="tenure",
        values="housing_units_estimated",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

acs_age_group.columns.name = None

acs_age_group = acs_age_group.rename(columns={
    "owner": "owner_units_estimated",
    "renter": "renter_units_estimated",
})

for col in ["owner_units_estimated", "renter_units_estimated"]:
    if col not in acs_age_group.columns:
        acs_age_group[col] = 0

acs_age_group["total_age_group_units"] = (
    acs_age_group["owner_units_estimated"] +
    acs_age_group["renter_units_estimated"]
)

acs_age_group["age_group_share_of_total_units"] = (
    acs_age_group["total_age_group_units"] /
    acs_age_group["total_occupied_units"]
)

acs_age_group["owner_rate_within_age_group"] = np.where(
    acs_age_group["total_age_group_units"] > 0,
    acs_age_group["owner_units_estimated"] / acs_age_group["total_age_group_units"],
    np.nan
)

acs_age_group["renter_rate_within_age_group"] = np.where(
    acs_age_group["total_age_group_units"] > 0,
    acs_age_group["renter_units_estimated"] / acs_age_group["total_age_group_units"],
    np.nan
)

acs_age_group = acs_age_group[
    [
        "county_clean",
        "county_fips_full",
        "year",
        "age_group",
        "owner_units_estimated",
        "renter_units_estimated",
        "total_age_group_units",
        "age_group_share_of_total_units",
        "owner_rate_within_age_group",
        "renter_rate_within_age_group",
    ]
]

acs_age_group = acs_age_group.sort_values(
    ["year", "county_clean", "age_group"]
).reset_index(drop=True)

print("Shape:", acs_age_group.shape)

acs_age_group.head()


Shape: (2646, 10)


,county_clean,county_fips_full,year,age_group,owner_units_estimated,renter_units_estimated,total_age_group_units,age_group_share_of_total_units,owner_rate_within_age_group,renter_rate_within_age_group
0,Atlantic,34001,2005,15-24,980.0,3410.0,4390.0,0.043215,0.223235,0.776765
1,Atlantic,34001,2005,25-34,7452.0,6992.0,14444.0,0.142188,0.515924,0.484076
2,Atlantic,34001,2005,35-44,14735.0,7108.0,21843.0,0.215024,0.674587,0.325413
3,Atlantic,34001,2005,45-54,16400.0,6368.0,22768.0,0.224130,0.720309,0.279691
4,Atlantic,34001,2005,55-59,6754.0,1624.0,8378.0,0.082474,0.806159,0.193841


### Step 9: Finalizing ACS Age Group Data with CBSA Codes

This groups New Jersey counties into CBSAs. The CBSA totals only include New Jersey counties, not the out-of-state counties.


In [46]:
county_to_cbsa = {
    "Atlantic": ("12100", "Atlantic City-Hammonton, NJ"),
    "Bergen": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Burlington": ("37980", "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD"),
    "Camden": ("37980", "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD"),
    "Cape May": ("36140", "Ocean City, NJ"),
    "Cumberland": ("47220", "Vineland-Bridgeton, NJ"),
    "Essex": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Gloucester": ("37980", "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD"),
    "Hudson": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Hunterdon": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Mercer": ("45940", "Trenton-Princeton, NJ"),
    "Middlesex": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Monmouth": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Morris": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Ocean": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Passaic": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Salem": ("37980", "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD"),
    "Somerset": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Sussex": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Union": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Warren": ("10900", "Allentown-Bethlehem-Easton, PA-NJ"),
}

acs_age_group["cbsa_code"] = acs_age_group["county_clean"].map(lambda x: county_to_cbsa[x][0])
acs_age_group["cbsa_name"] = acs_age_group["county_clean"].map(lambda x: county_to_cbsa[x][1])

acs_age_group_cbsa = (
    acs_age_group
    .groupby(["cbsa_code", "cbsa_name", "year", "age_group"], as_index=False)
    .agg({
        "owner_units_estimated": "sum",
        "renter_units_estimated": "sum",
        "total_age_group_units": "sum",
    })
)

cbsa_year_totals = (
    acs_age_group_cbsa
    .groupby(["cbsa_code", "year"])["total_age_group_units"]
    .transform("sum")
)

acs_age_group_cbsa["age_group_share_of_total_units"] = (
    acs_age_group_cbsa["total_age_group_units"] / cbsa_year_totals
)

acs_age_group_cbsa["owner_rate_within_age_group"] = np.where(
    acs_age_group_cbsa["total_age_group_units"] > 0,
    acs_age_group_cbsa["owner_units_estimated"] / acs_age_group_cbsa["total_age_group_units"],
    np.nan
)

acs_age_group_cbsa["renter_rate_within_age_group"] = np.where(
    acs_age_group_cbsa["total_age_group_units"] > 0,
    acs_age_group_cbsa["renter_units_estimated"] / acs_age_group_cbsa["total_age_group_units"],
    np.nan
)

acs_age_group_cbsa = acs_age_group_cbsa[
    [
        "cbsa_code",
        "cbsa_name",
        "year",
        "age_group",
        "owner_units_estimated",
        "renter_units_estimated",
        "total_age_group_units",
        "age_group_share_of_total_units",
        "owner_rate_within_age_group",
        "renter_rate_within_age_group",
    ]
]

acs_age_group_cbsa = acs_age_group_cbsa.sort_values(
    ["year", "cbsa_name", "age_group"]
).reset_index(drop=True)

print("Shape:", acs_age_group_cbsa.shape)

acs_age_group_cbsa.head()


Shape: (882, 10)


,cbsa_code,cbsa_name,year,age_group,owner_units_estimated,renter_units_estimated,total_age_group_units,age_group_share_of_total_units,owner_rate_within_age_group,renter_rate_within_age_group
0,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,15-24,153.0,181.0,334.0,0.008267,0.458084,0.541916
1,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,25-34,3577.0,2418.0,5995.0,0.148391,0.596664,0.403336
2,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,35-44,7083.0,2849.0,9932.0,0.245842,0.713149,0.286851
3,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,45-54,8140.0,2477.0,10617.0,0.262797,0.766695,0.233305
4,10900,"Allentown-Bethlehem-Easton, PA-NJ",2005,55-59,2918.0,1048.0,3966.0,0.098168,0.735754,0.264246


### Step 10: Finalizing ACS County-Year Data

This creates the county-year ACS file for the main merge. It keeps the 21 New Jersey counties.


In [47]:
acs_county_year = acs[acs["state_fips"].astype(str).str.zfill(2) == "34"].copy()

acs_county_year = acs_county_year[
    [
        "county_fips_full",
        "county_clean",
        "year",
        "total_occupied_units",
        "owner_occupied_total",
        "renter_occupied_total",
        "owner_rate",
        "renter_rate",
    ]
]

acs_county_year = acs_county_year.rename(columns={
    "county_fips_full": "county_fips",
    "county_clean": "county"
})

acs_county_year["county_fips"] = acs_county_year["county_fips"].astype(str).str.zfill(5)

acs_county_year = acs_county_year.sort_values(
    ["year", "county"]
).reset_index(drop=True)

print("Shape:", acs_county_year.shape)
print("Counties:", acs_county_year["county"].nunique())
print("Duplicates:", acs_county_year.duplicated(subset=["county_fips", "year"]).sum())

acs_county_year.head()


Shape: (294, 8)
Counties: 21
Duplicates: 0


,county_fips,county,year,total_occupied_units,owner_occupied_total,renter_occupied_total,owner_rate,renter_rate
0,34001,Atlantic,2005,101584,68970,32614,0.678946,0.321054
1,34003,Bergen,2005,332223,225175,107048,0.677783,0.322217
2,34005,Burlington,2005,164046,129201,34845,0.787590,0.212410
3,34007,Camden,2005,190659,130576,60083,0.684867,0.315133
4,34009,Cape May,2005,43616,34101,9515,0.781846,0.218154


### Step 11: Saving Data

This step saves the two cleaned ACS datasets.


In [48]:
acs_age_group_cbsa.to_csv(
    clean_path / "ACS_generation.csv",
    index=False
)

acs_county_year.to_csv(
    clean_path / "ACS_county.csv",
    index=False
)

